In [1]:
import io
import os
import pathlib
import pickle
from functools import partial as bind
from typing import Any, Callable

import flax.linen as nn
import jax
import jax.numpy as jnp
import numpy as np

import scipy

In [2]:
class InceptionV3(nn.Module):

  num_classes: int = 0

  @nn.compact
  def __call__(self, x, train=True, rng=jax.random.PRNGKey(0)):
    avg_pool = bind(nn.avg_pool, count_include_pad=False)
    x = Conv2D(32, 3, 2, name='Conv2d_1a_3x3')(x, train)
    x = Conv2D(32, 3, name='Conv2d_2a_3x3')(x, train)
    x = Conv2D(64, 3, pad='same', name='Conv2d_2b_3x3')(x, train)
    x = nn.max_pool(x, (3, 3), (2, 2))
    x = Conv2D(80, 1, name='Conv2d_3b_1x1')(x, train)
    x = Conv2D(192, 3, name='Conv2d_4a_3x3')(x, train)
    x = nn.max_pool(x, (3, 3), (2, 2))
    x = InceptionA(32, name='Mixed_5b')(x, train)
    x = InceptionA(64, name='Mixed_5c')(x, train)
    x = InceptionA(64, name='Mixed_5d')(x, train)
    x = InceptionB(name='Mixed_6a')(x, train)
    x = InceptionC(128, name='Mixed_6b')(x, train)
    x = InceptionC(160, name='Mixed_6c')(x, train)
    x = InceptionC(160, name='Mixed_6d')(x, train)
    x = InceptionC(192, name='Mixed_6e')(x, train)
    x = InceptionD(name='Mixed_7a')(x, train)
    x = InceptionE(avg_pool, name='Mixed_7b')(x, train)
    x = InceptionE(nn.max_pool, name='Mixed_7c')(x, train)
    x = x.mean((1, 2), keepdims=True)
    if self.num_classes:
      x = nn.Dropout(rate=0.5)(x, deterministic=not train, rng=rng)
      x = x.reshape((x.shape[0], -1))
      x = nn.Dense(self.num_classes, name='fc')(x)
    return x


class InceptionA(nn.Module):

  pool_depth: int = 0

  @nn.compact
  def __call__(self, x, train=True):
    a = Conv2D(64, 1, name='branch1x1')(x, train)
    b = Conv2D(48, 1, name='branch5x5_1')(x, train)
    b = Conv2D(64, 5, pad='same', name='branch5x5_2')(b, train)
    c = Conv2D(64, 1, name='branch3x3dbl_1')(x, train)
    c = Conv2D(96, 3, pad='same', name='branch3x3dbl_2')(c, train)
    c = Conv2D(96, 3, pad='same', name='branch3x3dbl_3')(c, train)
    d = nn.avg_pool(x, (3, 3), padding='same', count_include_pad=False)
    d = Conv2D(self.pool_depth, 1, name='branch_pool')(d, train)
    return jnp.concatenate((a, b, c, d), axis=-1)


class InceptionB(nn.Module):

  @nn.compact
  def __call__(self, x, train=True):
    a = Conv2D(384, 3, 2, name='branch3x3')(x, train)
    b = Conv2D(64, 1, name='branch3x3dbl_1')(x, train)
    b = Conv2D(96, 3, pad='same', name='branch3x3dbl_2')(b, train)
    b = Conv2D(96, 3, 2, name='branch3x3dbl_3')(b, train)
    c = nn.max_pool(x, (3, 3), (2, 2))
    return jnp.concatenate((a, b, c), axis=-1)


class InceptionC(nn.Module):

  depth: int = 0

  @nn.compact
  def __call__(self, x, train=True):
    a = Conv2D(192, 1, name='branch1x1')(x, train)
    b = Conv2D(self.depth, 1, name='branch7x7_1')(x, train)
    b = Conv2D(self.depth, (1, 7), pad='same', name='branch7x7_2')(b, train)
    b = Conv2D(192, (7, 1), pad='same', name='branch7x7_3')(b, train)
    c = Conv2D(self.depth, 1, name='branch7x7dbl_1')(x, train)
    c = Conv2D(self.depth, (7, 1), pad='same', name='branch7x7dbl_2')(c, train)
    c = Conv2D(self.depth, (1, 7), pad='same', name='branch7x7dbl_3')(c, train)
    c = Conv2D(self.depth, (7, 1), pad='same', name='branch7x7dbl_4')(c, train)
    c = Conv2D(192, (1, 7), pad='same', name='branch7x7dbl_5')(c, train)
    d = nn.avg_pool(x, (3, 3), padding='same', count_include_pad=False)
    d = Conv2D(192, 1, name='branch_pool')(d, train)
    return jnp.concatenate((a, b, c, d), axis=-1)


class InceptionD(nn.Module):

  @nn.compact
  def __call__(self, x, train=True):
    a = Conv2D(192, 1, name='branch3x3_1')(x, train)
    a = Conv2D(320, 3, 2, name='branch3x3_2')(a, train)
    b = Conv2D(192, 1, name='branch7x7x3_1')(x, train)
    b = Conv2D(192, (1, 7), pad='same', name='branch7x7x3_2')(b, train)
    b = Conv2D(192, (7, 1), pad='same', name='branch7x7x3_3')(b, train)
    b = Conv2D(192, 3, 2, name='branch7x7x3_4')(b, train)
    c = nn.max_pool(x, (3, 3), (2, 2))
    return jnp.concatenate((a, b, c), axis=-1)


class InceptionE(nn.Module):

  pool: Callable

  @nn.compact
  def __call__(self, x, train=True):
    a = Conv2D(320, 1, name='branch1x1')(x, train)
    b = Conv2D(384, 1, name='branch3x3_1')(x, train)
    b1 = Conv2D(384, (1, 3), pad='same', name='branch3x3_2a')(b, train)
    b2 = Conv2D(384, (3, 1), pad='same', name='branch3x3_2b')(b, train)
    b = jnp.concatenate((b1, b2), axis=-1)
    c = Conv2D(448, 1, name='branch3x3dbl_1')(x, train)
    c = Conv2D(384, 3, pad='same', name='branch3x3dbl_2')(c, train)
    c1 = Conv2D(384, (1, 3), pad='same', name='branch3x3dbl_3a')(c, train)
    c2 = Conv2D(384, (3, 1), pad='same', name='branch3x3dbl_3b')(c, train)
    c = jnp.concatenate((c1, c2), axis=-1)
    d = self.pool(x, (3, 3), padding='same')
    d = Conv2D(192, 1, name='branch_pool')(d, train)
    return jnp.concatenate((a, b, c, d), axis=-1)


class Conv2D(nn.Module):

  depth: int
  kernel: Any = 3
  stride: Any = 1
  pad: Any = 'valid'

  @nn.compact
  def __call__(self, x, train=True):
    kernel = self.kernel
    if isinstance(kernel, int):
      kernel = (kernel,) * 2
    args = (self.depth, kernel, self.stride, self.pad)
    x = nn.Conv(*args, use_bias=False, name='conv')(x)
    x = nn.BatchNorm(
        use_running_average=(not train),
        epsilon=1e-3, name='bn')(x)
    x = jax.nn.relu(x)
    return x


class FID:

  def __init__(
      self, weights, reference=None, resize=299,
      check_shapes=False, weights_device=None):
    if isinstance(weights, str):
      weights = pathlib.Path(weights)
    with weights.open('rb') as f:
      loaded = pickle.loads(f.read())
    if reference is not None:
      if isinstance(reference, str):
        reference = pathlib.Path(reference)
      with reference.open('rb') as f:
        reference = np.load(io.BytesIO(f.read()))
        self.ref = (reference['mu'], reference['sigma'])
        self.ref = jax.device_put(self.ref)
    else:
      self.ref = None
    model = InceptionV3()
    loaded = self._convert(loaded)
    if check_shapes:
      print('Checking FID weights...')
      reference = model.init(jax.random.PRNGKey(0), jnp.ones((1, 299, 299, 3)))
      pairs = jax.tree.map(lambda x, y: (x, y), loaded, reference)
      for path, (x1, x2) in jax.tree_util.tree_leaves_with_path(
          pairs, is_leaf=lambda x: isinstance(x, tuple) and len(x) == 2):
        name = tuple(segment.key for segment in path)
        assert x1.shape == x2.shape, (name, x1.shape, x2.shape)
    self.params = jax.device_put(loaded, weights_device)
    self.apply = bind(jax.jit(bind(model.apply, train=False)), self.params)
    self.resize = resize

  def compute_acts(self, imgs):
    assert imgs.dtype == jnp.uint8, imgs.dtype
    assert imgs.ndim == 4, imgs.shape
    imgs = imgs.astype(jnp.float32) / 255
    imgs = imgs * 2 - 1
    if self.resize:
      imgs = jax.image.resize(
          imgs, ((len(imgs)), self.resize, self.resize, 3),
          method='bilinear', antialias=False)
    acts = self.apply(imgs)
    acts = jnp.squeeze(acts, (1, 2))
    return acts

  def compute_stats(self, acts):
    if isinstance(acts, (list, tuple)):
      acts = jnp.concatenate(acts, 0)
    assert acts.shape[1:] == (2048,)
    mu = acts.mean(0)
    sigma = jnp.cov(acts, rowvar=False)
    return (mu, sigma)

  def compute_score(self, stats, ref=None):
    stats_cpu = jax.device_put(stats, device=jax.devices("cpu")[0])
    ref_cpu = jax.device_put(self.ref if ref is None else ref, device=jax.devices("cpu")[0])
    
    mu1, sigma1 = stats_cpu
    mu2, sigma2 = ref_cpu
    diff = mu1 - mu2
    offset = jnp.eye(sigma1.shape[0]) * 1e-6
    covmean = scipy.linalg.sqrtm((sigma1 + offset) @ (sigma2 + offset))
    covmean = np.real(covmean)
    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)

    return fid

  def _convert(self, params):
    params = {'batch_stats': {}, 'params': params}
    params['params'].pop('fc')  # FID doesn't use the output layer.
    # Move BN stats into separate collection for Flax.
    is_layer = lambda x: isinstance(x, dict) and 'bn' in x
    for path, layer in jax.tree_util.tree_leaves_with_path(
        params['params'], is_leaf=is_layer):
      target = params['batch_stats']
      for segment in path:
        if segment.key not in target:
          target[segment.key] = {}
        target = target[segment.key]
      mean = layer['bn'].pop('mean')
      var = layer['bn'].pop('var')
      target['bn'] = {'mean': mean, 'var': var}
    return params

In [3]:
if True:
  import tqdm

  weights = '/home/yixiuz/inception_v3_weights_fid.pickle?dl=1'
  reference = '/home/yixiuz/VIRTUAL_imagenet256_labeled.npz'
  fid = FID(weights, reference)


/tmp/ipykernel_126839/3506921881.py:147: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  loaded = pickle.loads(f.read())


In [26]:
# path = "/home/yixiuz/4psteps_1gibbs_k=32_late_entry_images"
path = "/home/yixiuz/8psteps_1gibbs_k=16_images"
npz_files = [f for f in os.listdir(path) if f.endswith('.npz')]

arrays = []

# Loop through and load each file
for npz_file in npz_files:
    data = np.load(os.path.join(path, npz_file))
    
    # Assuming each .npz file has a single array
    # If multiple arrays exist, specify the correct key
    for key in data.files:
        arrays.append(data[key])

# Concatenate all arrays along the first axis (modify if needed)
samples = np.concatenate(arrays, axis=0)

print("Shape of concatenated array:", samples.shape)

Shape of concatenated array: (10112, 256, 256, 3)


In [27]:
batch = 64
acts = []
for i in tqdm.trange(0, len(samples), batch):
    acts.append(fid.compute_acts(samples[i: i + batch]))
assert sum(len(x) for x in acts) == len(samples)

print('Computing score...')
stats = fid.compute_stats(acts)
score = fid.compute_score(stats)

print(score)  # 3.9371355

100%|██████████| 158/158 [00:21<00:00,  7.30it/s]


Computing score...
8.806374


In [28]:
sample_acts = jnp.concatenate(acts)
sample_acts.shape

(10112, 2048)

# Inception Score

In [17]:
import jax.scipy.special

class InceptionScore:

    def __init__(self, weights, resize=299, weights_device=None):
        if isinstance(weights, str):
            weights = pathlib.Path(weights)
        with weights.open('rb') as f:
            loaded = pickle.loads(f.read())

        # Weights have 1008 classes, this is a bug so we're gonna follow this if we want to compare with previous work
        model = InceptionV3(num_classes=1008)
        loaded = self._convert(loaded)

        self.params = jax.device_put(loaded, weights_device)
        self.apply = bind(jax.jit(bind(model.apply, train=False)), self.params)
        self.resize = resize

    def compute_probs(self, imgs):
        """Compute softmax class probabilities for generated images."""
        assert imgs.dtype == jnp.uint8, imgs.dtype
        assert imgs.ndim == 4, imgs.shape
        imgs = imgs.astype(jnp.float32) / 255  # Normalize images to [0,1]
        imgs = imgs * 2 - 1  # Scale to [-1,1] as expected by InceptionV3
        
        if self.resize:
            imgs = jax.image.resize(
                imgs, ((len(imgs)), self.resize, self.resize, 3),
                method='bilinear', antialias=False)
        
        logits = self.apply(imgs)  # Get model logits
        probs = jax.nn.softmax(logits, axis=-1)  # Convert to probabilities
        return probs

    def compute_inception_score(self, probs, splits=10):
        """Compute the Inception Score (IS) from softmax class probabilities."""
        N = probs.shape[0]
        split_scores = []

        for i in range(splits):
            part = probs[i * (N // splits): (i + 1) * (N // splits), :]
            p_yx = part
            p_y = jnp.mean(part, axis=0, keepdims=True)
            
            kl_div = jnp.sum(p_yx * (jnp.log(p_yx + 1e-6) - jnp.log(p_y + 1e-6)), axis=1)
            split_scores.append(jnp.exp(jnp.mean(kl_div)))

        return jnp.mean(jnp.array(split_scores)), jnp.std(jnp.array(split_scores))

    def _convert(self, params):
        """Convert stored parameters for compatibility with Flax."""
        params = {'batch_stats': {}, 'params': params}
        # Move BN stats into separate collection for Flax
        is_layer = lambda x: isinstance(x, dict) and ('bn' in x)
        for path, layer in jax.tree.leaves_with_path(params['params'], is_leaf=is_layer):
            # There's a bug somewhere lol
            if not is_layer(layer):
                continue
            
            target = params['batch_stats']
            for segment in path:
                if segment.key not in target:
                    target[segment.key] = {}
                target = target[segment.key]
            mean = layer['bn'].pop('mean')
            var = layer['bn'].pop('var')
            target['bn'] = {'mean': mean, 'var': var}
        return params


In [19]:
# weights = pathlib.Path(weights)
# with weights.open('rb') as f:
#     loaded = pickle.loads(f.read())

# import pprint
# pprint.pprint(loaded)

In [13]:
# Read all images
path = "/home/yixiuz/samples/8psteps_1gibbs_size=-1.0_k=16_images"
npz_files = [f for f in os.listdir(path) if f.endswith('.npz')]

samples=[]
for _, npz_file in tqdm.tqdm(enumerate(npz_files)):
    data = np.load(os.path.join(path, npz_file))
    for key in data.files:
        samples.append(data[key])

# Concatenate all arrays along the first axis (modify if needed)
samples = np.concatenate(samples, axis=0)

40it [01:30,  2.27s/it]


In [18]:
# Load InceptionV3 weights
# Initialize Inception Score computation
weights = '/home/yixiuz/inception_v3_weights_fid.pickle?dl=1'
inception_score = InceptionScore(weights)


# Compute softmax probabilities
batch = 128
probs = []

compute_prob = jax.jit(inception_score.compute_probs)

for i in tqdm.trange(0, len(samples), batch):
    probs.append(compute_prob(samples[i: i + batch]))

probs = jnp.concatenate(probs, axis=0)

# Compute the final Inception Score (mean ± std)
# Since we're in imagenet, we only use one split
mean_IS, std_IS = inception_score.compute_inception_score(probs, splits=1)
print(f'Inception Score: {mean_IS:.4f} ± {std_IS:.4f}')


/tmp/ipykernel_126839/3090380092.py:9: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  loaded = pickle.loads(f.read())
  0%|          | 0/391 [00:00<?, ?it/s]

100%|██████████| 391/391 [01:05<00:00,  5.99it/s]


Inception Score: 185.9071 ± 0.0000


# Precision and recall

In [4]:
import jax
import jax.numpy as jnp
import numpy as np
from time import time

# ----------------------------------------------------------------------------
# Compute pairwise distances in JAX
# ----------------------------------------------------------------------------

@jax.jit
def batch_pairwise_distances(U, V):
    """Compute pairwise squared Euclidean distances between two batches of feature vectors."""
    norm_u = jnp.sum(jnp.square(U), axis=1, keepdims=True)
    norm_v = jnp.sum(jnp.square(V), axis=1, keepdims=True).T
    D = jnp.maximum(norm_u - 2 * jnp.dot(U, V.T) + norm_v, 0.0)  # Ensure non-negative distances
    return D


# ----------------------------------------------------------------------------
# JAX Version of Manifold Estimation
# ----------------------------------------------------------------------------

class ManifoldEstimator:
    """Estimates the manifold of given feature vectors."""

    def __init__(self, features, row_batch_size=25000, col_batch_size=50000, nhood_sizes=[3], 
                 clamp_to_percentile=None, eps=1e-5):
        """Estimate the manifold of given feature vectors.
        
        Args:
            features (np.array): Matrix of feature vectors to estimate their manifold.
            row_batch_size (int): Row batch size for pairwise distance computation.
            col_batch_size (int): Column batch size for pairwise distance computation.
            nhood_sizes (list): Number of neighbors used to estimate the manifold.
            clamp_to_percentile (float): Prune hyperspheres that have radius larger than
                the given percentile.
            eps (float): Small number for numerical stability.
        """
        num_images = features.shape[0]
        self.nhood_sizes = nhood_sizes
        self.num_nhoods = len(nhood_sizes)
        self.eps = eps
        self.row_batch_size = row_batch_size
        self.col_batch_size = col_batch_size
        self._ref_features = features

        # Compute k-nearest neighbor distances
        self.D = np.zeros([num_images, self.num_nhoods], dtype=np.float32)
        distance_batch = np.zeros([row_batch_size, num_images], dtype=np.float32)
        seq = np.arange(max(self.nhood_sizes) + 1, dtype=np.int32)

        for begin1 in range(0, num_images, row_batch_size):
            end1 = min(begin1 + row_batch_size, num_images)
            row_batch = features[begin1:end1]

            for begin2 in range(0, num_images, col_batch_size):
                end2 = min(begin2 + col_batch_size, num_images)
                col_batch = features[begin2:end2]

                # Compute pairwise distances between row and column batches
                distance_batch[:end1-begin1, begin2:end2] = batch_pairwise_distances(row_batch, col_batch)

            # Find k-nearest neighbor distances
            self.D[begin1:end1, :] = np.partition(distance_batch[:end1-begin1, :], seq, axis=1)[:, self.nhood_sizes]

        if clamp_to_percentile is not None:
            max_distances = np.percentile(self.D, clamp_to_percentile, axis=0)
            self.D[self.D > max_distances] = 0

    def evaluate(self, eval_features, return_realism=False, return_neighbors=False):
        """Evaluate if new feature vectors are inside the manifold."""
        num_eval_images = eval_features.shape[0]
        num_ref_images = self.D.shape[0]
        distance_batch = np.zeros([self.row_batch_size, num_ref_images], dtype=np.float32)
        batch_predictions = np.zeros([num_eval_images, self.num_nhoods], dtype=np.int32)
        max_realism_score = np.zeros([num_eval_images], dtype=np.float32)
        nearest_indices = np.zeros([num_eval_images], dtype=np.int32)

        for begin1 in range(0, num_eval_images, self.row_batch_size):
            end1 = min(begin1 + self.row_batch_size, num_eval_images)
            feature_batch = eval_features[begin1:end1]

            for begin2 in range(0, num_ref_images, self.col_batch_size):
                end2 = min(begin2 + self.col_batch_size, num_ref_images)
                ref_batch = self._ref_features[begin2:end2]

                distance_batch[:end1-begin1, begin2:end2] = batch_pairwise_distances(feature_batch, ref_batch)

            # Check if samples are within the estimated manifold
            samples_in_manifold = distance_batch[:end1-begin1, :, None] <= self.D
            batch_predictions[begin1:end1] = np.any(samples_in_manifold, axis=1).astype(np.int32)

            max_realism_score[begin1:end1] = np.max(self.D[:, 0] / (distance_batch[:end1-begin1, :] + self.eps), axis=1)
            nearest_indices[begin1:end1] = np.argmin(distance_batch[:end1-begin1, :], axis=1)

        if return_realism and return_neighbors:
            return batch_predictions, max_realism_score, nearest_indices
        elif return_realism:
            return batch_predictions, max_realism_score
        elif return_neighbors:
            return batch_predictions, nearest_indices

        return batch_predictions


# ----------------------------------------------------------------------------
# Compute k-NN Precision and Recall
# ----------------------------------------------------------------------------

def knn_precision_recall_features(ref_features, eval_features, nhood_sizes=[3], 
                                  row_batch_size=10000, col_batch_size=50000):
    """Calculates k-NN precision and recall for two sets of feature vectors.
    
    Args:
        ref_features (np.array): Feature vectors of reference images.
        eval_features (np.array): Feature vectors of generated images.
        nhood_sizes (list): Number of neighbors used to estimate the manifold.
        row_batch_size (int): Row batch size to compute pairwise distances.
        col_batch_size (int): Column batch size to compute pairwise distances.

    Returns:
        dict: Contains precision and recall calculated from ref_features and eval_features.
    """
    state = {}
    num_images = ref_features.shape[0]

    # Estimate manifolds
    ref_manifold = ManifoldEstimator(ref_features, row_batch_size, col_batch_size, nhood_sizes) 
    eval_manifold = ManifoldEstimator(eval_features, row_batch_size, col_batch_size, nhood_sizes)

    print(f'Evaluating k-NN precision and recall with {num_images} samples...')
    start = time()

    # Compute precision (how many eval samples lie in the ref manifold)
    precision = ref_manifold.evaluate(eval_features)
    state['precision'] = precision.mean(axis=0)

    # Compute recall (how many ref samples lie in the eval manifold)
    recall = eval_manifold.evaluate(ref_features)
    state['recall'] = recall.mean(axis=0)

    print(f'Evaluated k-NN precision and recall in: {time() - start:.4f}s')

    return state, ref_manifold, eval_manifold


In [8]:
ref_acts = jnp.load("/home/yixiuz/imagenet_val_activations.npz")['activations']
sample_acts = jnp.load("/home/yixiuz/samples/8psteps_1gibbs_size=-1.0_k=16_acts.npy")[:50000]

In [7]:
# ref_acts.shape
sample_acts.shape

(50048, 2048)

In [9]:
# ref_features = ref_acts[:5000]
# ref_manifold = ManifoldEstimator(ref_features) 

In [9]:
# np.sum(ref_manifold.D == 0)

In [10]:
# Generate dummy reference and evaluation features
# ref_features = np.random.randn(50000, 512)  # Reference feature vectors
ref_features = ref_acts
# eval_features = np.random.randn(50000, 512) / 2 # Generated feature vectors
eval_features = sample_acts

# Compute k-NN precision and recall
metrics, ref_mani, eval_mani = knn_precision_recall_features(ref_features, eval_features)

# Print results
print(f'Precision: {metrics["precision"]}')
print(f'Recall: {metrics["recall"]}')


Evaluating k-NN precision and recall with 50000 samples...
Evaluated k-NN precision and recall in: 54.4807s
Precision: [0.77638]
Recall: [0.45206]
